## import Library

In [4]:
import os
import cv2
import shutil
import random

## Config

In [ ]:
INPUT_PATH = "RM_PROJECT/Dataset"
OUTPUT_PATH = "processed_dataset"
IMG_SIZE = (224, 224)

## Data Cleaning

In [6]:
def clean_dataset(path):
    print("Cleaning dataset...")
    for folder in ['cancer', 'normal']:
        folder_path = os.path.join(path, folder)
        
        for file in os.listdir(folder_path):
            img_path = os.path.join(folder_path, file)
            img = cv2.imread(img_path)
            
            if img is None:
                print(f"Removing corrupted file: {img_path}")
                os.remove(img_path)

## Resize + RBG + Enhancement

In [7]:
def process_images(input_path, output_path):
    print("Processing images...")
    
    for folder in ['cancer', 'normal']:
        input_folder = os.path.join(input_path, folder)
        output_folder = os.path.join(output_path, folder)
        
        os.makedirs(output_folder, exist_ok=True)
        
        for file in os.listdir(input_folder):
            img_path = os.path.join(input_folder, file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                continue
            
            # CLAHE (Enhancement)
            clahe = cv2.createCLAHE(clipLimit=2.0)
            img = clahe.apply(img)
            
            # Resize
            img = cv2.resize(img, IMG_SIZE)
            
            # Convert ke RGB
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
            
            cv2.imwrite(os.path.join(output_folder, file), img)

## Data Splitting

In [8]:
def split_data(input_path, output_path, train=0.7, val=0.15, test=0.15):
    print("Splitting dataset...")
    
    for folder in ['cancer', 'normal']:
        files = os.listdir(os.path.join(input_path, folder))
        random.shuffle(files)
        
        n = len(files)
        train_end = int(train * n)
        val_end = int((train + val) * n)
        
        splits = {
            'train': files[:train_end],
            'val': files[train_end:val_end],
            'test': files[val_end:]
        }
        
        for split in splits:
            split_folder = os.path.join(output_path, split, folder)
            os.makedirs(split_folder, exist_ok=True)
            
            for file in splits[split]:
                src = os.path.join(input_path, folder, file)
                dst = os.path.join(split_folder, file)
                shutil.copy(src, dst)

## Main

In [9]:
if __name__ == "__main__":
    
    # Step 1: Cleaning
    clean_dataset(INPUT_PATH)
    
    # Step 2: Processing
    process_images(INPUT_PATH, OUTPUT_PATH)
    
    # Step 3: Splitting
    split_data(OUTPUT_PATH, OUTPUT_PATH)
    
    print("Preprocessing selesai!")

Cleaning dataset...
Processing images...
Splitting dataset...
Preprocessing selesai!
